Utilizzando lo script della lezione, esegui il codice due volte separatemente (riavvinado il kernel o lo script)
Verifica che la final_loss sia identica in entrambe le esecuzioni fino al decimo decimale.
Successivamente, commenta la righe int-reproducibility(42) e riesegui il test due volte: osserva come le perdite finali divergano a causa dell'inizzializzazione casuale dei pesi.
Modifica il batch_size portandolo a 1024 e monitora (se possibile tramite TaskManager) della GPU e log distema come l'occupazione della VRMA aumenti rispetto a un batch di 64

In [ ]:
import os

# 1. SET BACKEND (Deve essere fatto prima di importare Keras)
os.environ["KERAS_BACKEND"] = "torch"
 
import keras 
from keras import layers, models, mixed_precision
import random
import numpy as np
import tensorflow as tf


# 1. IL PILASTRO DELLA RIPRODUCIBILITÀ (SEEDING)
def init_reproducibility(seed=42):
    """
    Imposta il seme aleatorio su tutti i livelli del software stack.
    Teoria: L'inizializzazione dei pesi (es. Glorot) usa distribuzioni 
    probabilistiche. Fissando il seed, la 'casuallità' diventa deterministica.
    """
    # Seme per le operazioni native di Python (es. shuffle di liste)
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    
    # Seme per Numpy (gestione dei vettori e preprocessing)
    np.random.seed(seed)
    
    # Seme globale di TensorFlow (inizializzazione pesi e dropout)
    # Nel 2026, questa funzione sincronizza anche i seed dei generatori interni
    tf.keras.utils.set_random_seed(seed)
    
    # Forza TensorFlow a usare operazioni deterministiche su GPU (se disponibili)
    # Nota: Questo può rallentare leggermente il training ma garantisce l'identità dei bit
    tf.config.experimental.enable_op_determinism()
    print(f"[*] Ambiente impostato con Seed: {seed}")

# 2. GESTIONE MEMORIA E COSTI (MIXED PRECISION)
def setup_optimization():
    """
    Abilita la Mixed Precision per dimezzare l'uso della VRAM.
    Teoria: Usa float16 per i calcoli e float32 per le variabili critiche.
    Riduce i costi cloud accelerando il training sulle GPU moderne.
    """
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print(f"[*] Ottimizzazione VRAM: {policy.compute_dtype} abilitata")

# 3. SCELTA DELL'ARCHITETTURA (CNN vs ANN)
def build_model(input_shape, num_classes, architecture_type="CNN"):
    """
    Dimostra la scelta del modello basata sul problema.
    CNN: Analisi spaziale (immagini).
    ANN: Analisi tabulare (dati piatti).
    """
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    if architecture_type == "CNN":
        # Scelta corretta per immagini: sfrutta l'invarianza per traslazione
        # Teoria: I filtri estraggono feature locali indipendentemente dalla posizione
        model.add(layers.Conv2D(32, (3, 3), activation='relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Flatten())
    else:
        # ANN Densa: Adatta per dati dove l'ordine dei pixel non è strutturato
        # Teoria: Ogni neurone è connesso a ogni input, ignorando la topologia 2D
        model.add(layers.Flatten())
        model.add(layers.Dense(128, activation='relu'))

    model.add(layers.Dense(num_classes, activation='softmax', dtype='float32'))
    return model

# --- ESECUZIONE PIPELINE ---
init_reproducibility(42)
setup_optimization()

# Caricamento dati (MNIST come esempio di benchmark)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train[..., np.newaxis] / 255.0, x_test[..., np.newaxis] / 255.0

# Costruzione del modello (Scelta: CNN per dati di tipo immagine)
model = build_model((28, 28, 1), 10, architecture_type="CNN")

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Definiamo un percorso con estensione .keras (Standard 2026)
checkpoint_path = "models/best_mnist_model.keras"

# Creiamo la cartella se non esiste
os.makedirs("models", exist_ok=True)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,      # Salva solo se l'accuratezza migliora
    monitor='val_accuracy',   # Monitora la precisione di validazione
    mode='max',
    verbose=1
)

# EarlyStopping: Best practice per evitare l'overfitting e risparmiare costi energetici
es_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Esegui il fit con i nuovi callback (aggiungendo validation_split)
model.fit(
    x_train, y_train, 
    epochs=10, 
    batch_size=64, 
    validation_split=0.1,  # Necessario per monitorare val_accuracy
    callbacks=[cp_callback, es_callback]
)

# val_accuracy: 0.9847 - val_loss: 0.0554

# Riproducibilità (init_reproducibility): L'utilizzo di un seed fisso (es. 42) blocca i generatori di numeri casuali di NumPy, PyTorch/TensorFlow e del sistema. 
# Questo garantisce che i pesi iniziali e lo shuffling del dataset siano identici tra le esecuzioni, portando a una final_loss perfettamente replicabile.
#
# Divergenza senza Seed: Rimuovendo il seed, ogni esecuzione parte da un punto diverso nello spazio dei parametri a causa dell'inizializzazione stocastica. 
# Di conseguenza, l'ottimizzatore seguirà traiettorie differenti, producendo risultati finali diversi.
#
# Batch Size e VRAM: Aumentando il batch_size (da 64 a 1024), la GPU deve caricare e processare un numero molto maggiore di immagini e attivazioni contemporaneamente. 
# Ciò causa un incremento diretto dell'occupazione della VRAM, poiché la memoria necessaria per i tensori intermedi durante il forward e backward pass scala quasi linearmente con la dimensione del batch.